# CP2 — Previsão de preço de criptomoeda com LSTM

**Moeda:** Bitcoin (USD) · **Fonte:** API pública do CoinGecko · **Modelo:** LSTM (TensorFlow/Keras)

**Integrante:** Rodrigo Leme — RM550266

Etapas do notebook:
1. Coleta dos dados (API CoinGecko)
2. Pré-processamento (resample, valores faltantes, normalização, janelas)
3. Construção e treino da LSTM (treino / validação / teste em ordem temporal)
4. Avaliação (MSE, RMSE, MAE + baseline de persistência)
5. Experimentos com diferentes janelas de tempo (`window_size`)
6. Predição em dados novos (nova chamada à API com os últimos 7 dias)

## 0. Bibliotecas e configurações

In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense

# Semente para os resultados serem reproduzíveis
tf.keras.utils.set_random_seed(42)

# Pasta onde os gráficos e o modelo serão salvos
os.makedirs("resultados", exist_ok=True)

COIN = "bitcoin"      # moeda escolhida
DAYS = 90             # dias de histórico (o CoinGecko devolve dados de hora em hora para até 90 dias)
DIAS_NOVOS = 7        # últimos 7 dias ficam fora do treino e são usados como "dados novos"
WINDOW_SIZE = 24      # usar as últimas 24 horas para prever a próxima hora
EPOCHS = 20
BATCH_SIZE = 32

## 1. Coleta dos dados
Função de coleta baseada no exemplo do enunciado: chama o endpoint `market_chart` do CoinGecko e devolve um DataFrame com data/hora e preço.

In [ ]:
def get_historical_data(coin="bitcoin", days=90):
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart?vs_currency=usd&days={days}"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    df = pd.DataFrame(data["prices"], columns=["Timestamp", "Price (USD)"])
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], unit="ms")
    return df

df_raw = get_historical_data(COIN, DAYS)
print(df_raw.shape)
df_raw.head()

Separação temporal dos dados: tudo antes de **(agora − 7 dias)** é usado no treino/validação/teste.
Os últimos 7 dias **não entram no treino** — eles serão buscados de novo na API na etapa 6.

In [ ]:
data_corte = df_raw["Timestamp"].max() - pd.Timedelta(days=DIAS_NOVOS)
df_hist = df_raw[df_raw["Timestamp"] <= data_corte]
print("Período usado no modelo:", df_hist["Timestamp"].min(), "até", df_hist["Timestamp"].max())

plt.figure(figsize=(12, 4))
plt.plot(df_raw["Timestamp"], df_raw["Price (USD)"], label="Preço")
plt.axvline(data_corte, color="red", linestyle="--", label="Corte (dados novos à direita)")
plt.title(f"Preço histórico do {COIN} (USD)")
plt.xlabel("Data")
plt.ylabel("Preço (USD)")
plt.legend()
plt.savefig("resultados/01_historico.png", bbox_inches="tight")
plt.show()

## 2. Pré-processamento
- **Resample** para frequência de 1 hora (média dos preços de cada hora)
- **Valores faltantes**: interpolação linear
- **Normalização** com `MinMaxScaler` (ajustado só nos dados de treino, para não vazar informação)
- **Janelas de tempo**: as últimas `window_size` horas viram a entrada e a hora seguinte vira o alvo

In [ ]:
def preprocessar(df):
    serie = df.set_index("Timestamp")["Price (USD)"]
    serie = serie.resample("1h").mean()   # frequência regular de 1 hora
    serie = serie.interpolate()            # preenche horas faltantes
    return serie

serie = preprocessar(df_hist)
print("Total de horas:", len(serie))
print("Valores faltantes após tratamento:", serie.isna().sum())

In [ ]:
def criar_janelas(dados, window_size):
    # dados: array normalizado (n, 1)
    X, y = [], []
    for i in range(window_size, len(dados)):
        X.append(dados[i - window_size:i])   # últimas window_size horas
        y.append(dados[i])                   # próxima hora
    return np.array(X), np.array(y)

Divisão **treino / validação / teste** respeitando a ordem do tempo (70% / 15% / 15%), sem embaralhar.

In [ ]:
def preparar_dados(serie, window_size):
    valores = serie.values.reshape(-1, 1)
    n = len(valores)
    fim_treino = int(n * 0.70)
    fim_val = int(n * 0.85)

    scaler = MinMaxScaler()
    scaler.fit(valores[:fim_treino])          # ajusta só no treino
    valores_norm = scaler.transform(valores)

    X, y = criar_janelas(valores_norm, window_size)
    # o alvo y[k] corresponde à posição k + window_size da série original
    pos = np.arange(window_size, n)
    treino = pos < fim_treino
    val = (pos >= fim_treino) & (pos < fim_val)
    teste = pos >= fim_val

    return (X[treino], y[treino]), (X[val], y[val]), (X[teste], y[teste]), scaler

(X_train, y_train), (X_val, y_val), (X_test, y_test), scaler = preparar_dados(serie, WINDOW_SIZE)
print("Treino:", X_train.shape, " Validação:", X_val.shape, " Teste:", X_test.shape)

## 3. Construção e treino do modelo
Arquitetura simples: uma camada **LSTM com 50 unidades** seguida de uma camada **Dense** com 1 saída (o preço da próxima hora).
Função de perda: MSE · Otimizador: Adam.

In [ ]:
def criar_modelo(window_size):
    modelo = Sequential([
        Input(shape=(window_size, 1)),
        LSTM(50),
        Dense(1)
    ])
    modelo.compile(optimizer="adam", loss="mse")
    return modelo

modelo = criar_modelo(WINDOW_SIZE)
modelo.summary()

In [ ]:
historico = modelo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

### Curvas de perda (loss / val_loss)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(historico.history["loss"], label="loss (treino)")
plt.plot(historico.history["val_loss"], label="val_loss (validação)")
plt.title("Curva de aprendizado da LSTM")
plt.xlabel("Época")
plt.ylabel("MSE (dados normalizados)")
plt.legend()
plt.savefig("resultados/02_loss.png", bbox_inches="tight")
plt.show()

## 4. Avaliação no conjunto de teste
As previsões voltam para dólar (`inverse_transform`) e são comparadas com os valores reais usando **MSE, RMSE e MAE**.
Também comparamos com o **baseline de persistência** (próximo preço = último preço conhecido).

In [ ]:
def calcular_metricas(real, previsto):
    mse = mean_squared_error(real, previsto)
    return {"MSE": mse, "RMSE": np.sqrt(mse), "MAE": mean_absolute_error(real, previsto)}

# Previsão da LSTM (em USD)
pred_test = scaler.inverse_transform(modelo.predict(X_test, verbose=0))
real_test = scaler.inverse_transform(y_test)

# Baseline de persistência: último valor de cada janela
pers_test = scaler.inverse_transform(X_test[:, -1, :])

resultados = pd.DataFrame({
    "LSTM": calcular_metricas(real_test, pred_test),
    "Persistência": calcular_metricas(real_test, pers_test)
}).T
resultados.round(2)

In [ ]:
datas_teste = serie.index[-len(real_test):]

plt.figure(figsize=(12, 4))
plt.plot(datas_teste, real_test, label="Real")
plt.plot(datas_teste, pred_test, label="Previsto (LSTM)")
plt.title("Conjunto de teste: previsão vs valores reais")
plt.xlabel("Data")
plt.ylabel("Preço (USD)")
plt.legend()
plt.savefig("resultados/03_teste_previsto_vs_real.png", bbox_inches="tight")
plt.show()

## 5. Experimentos com diferentes janelas de tempo
Treinamos o mesmo modelo com `window_size` de **12, 24 e 48 horas** e comparamos as métricas no teste.
Também incluímos os dois modelos simples sugeridos: **persistência** e **média móvel** (média da janela).

In [ ]:
JANELAS = [12, 24, 48]
linhas = []
historicos = {}

for w in JANELAS:
    tf.keras.utils.set_random_seed(42)
    (Xtr, ytr), (Xva, yva), (Xte, yte), sc = preparar_dados(serie, w)
    m = criar_modelo(w)
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    historicos[w] = h.history

    real = sc.inverse_transform(yte)
    lstm = sc.inverse_transform(m.predict(Xte, verbose=0))
    media_movel = sc.inverse_transform(Xte.mean(axis=1))
    persist = sc.inverse_transform(Xte[:, -1, :])

    linhas.append({"window_size": w, "modelo": "LSTM", **calcular_metricas(real, lstm)})
    linhas.append({"window_size": w, "modelo": "Média móvel", **calcular_metricas(real, media_movel)})
    linhas.append({"window_size": w, "modelo": "Persistência", **calcular_metricas(real, persist)})

experimentos = pd.DataFrame(linhas)
experimentos.to_csv("resultados/experimentos.csv", index=False)
experimentos.round(2)

In [ ]:
plt.figure(figsize=(8, 4))
for w in JANELAS:
    plt.plot(historicos[w]["val_loss"], label=f"window_size={w}")
plt.title("val_loss por tamanho de janela")
plt.xlabel("Época")
plt.ylabel("MSE (dados normalizados)")
plt.legend()
plt.savefig("resultados/04_experimentos_val_loss.png", bbox_inches="tight")
plt.show()

rmse_lstm = experimentos[experimentos["modelo"] == "LSTM"]
plt.figure(figsize=(6, 4))
plt.bar(rmse_lstm["window_size"].astype(str), rmse_lstm["RMSE"])
plt.title("RMSE da LSTM no teste por janela")
plt.xlabel("window_size (horas)")
plt.ylabel("RMSE (USD)")
plt.savefig("resultados/05_experimentos_rmse.png", bbox_inches="tight")
plt.show()

O modelo principal (seção 3, `window_size = 24`) é salvo para ficar disponível junto com o projeto.

In [ ]:
modelo.save("resultados/modelo_lstm.keras")

## 6. Predição em dados novos
Consumimos **novamente a API**, agora pedindo os **últimos 7 dias**. Esses dados ficaram fora do treino (estão depois da data de corte).

1. **Previsão passo a passo**: para cada hora nova, o modelo usa as 24 horas reais anteriores e prevê a próxima.
2. **Projeção futura**: a partir da última hora real, o modelo prevê as próximas 24 horas, usando cada previsão como entrada da seguinte.

In [ ]:
df_novo = get_historical_data(COIN, DIAS_NOVOS)
serie_nova = preprocessar(df_novo)
serie_nova = serie_nova[serie_nova.index > serie.index[-1]]   # só horas que não foram usadas
print("Horas novas:", len(serie_nova), "|", serie_nova.index.min(), "até", serie_nova.index.max())

In [ ]:
# Junta as últimas 24 h do histórico com os dados novos para montar as janelas
serie_completa = pd.concat([serie.iloc[-WINDOW_SIZE:], serie_nova])
valores_norm = scaler.transform(serie_completa.values.reshape(-1, 1))
X_novo, y_novo = criar_janelas(valores_norm, WINDOW_SIZE)

pred_novo = scaler.inverse_transform(modelo.predict(X_novo, verbose=0))
real_novo = scaler.inverse_transform(y_novo)
pers_novo = scaler.inverse_transform(X_novo[:, -1, :])

metricas_novos = pd.DataFrame({
    "LSTM": calcular_metricas(real_novo, pred_novo),
    "Persistência": calcular_metricas(real_novo, pers_novo)
}).T
metricas_novos.round(2)

In [ ]:
# Projeção das próximas 24 horas (cada previsão vira entrada da próxima)
PASSOS_FUTUROS = 24
janela = valores_norm[-WINDOW_SIZE:].copy()
futuro_norm = []
for _ in range(PASSOS_FUTUROS):
    proximo = modelo.predict(janela.reshape(1, WINDOW_SIZE, 1), verbose=0)
    futuro_norm.append(proximo[0])
    janela = np.vstack([janela[1:], proximo])

futuro = scaler.inverse_transform(np.array(futuro_norm))
datas_futuro = pd.date_range(serie_nova.index[-1] + pd.Timedelta(hours=1), periods=PASSOS_FUTUROS, freq="1h")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(serie_nova.index, real_novo, label="Real (dados novos)")
plt.plot(serie_nova.index, pred_novo, label="Previsto pela LSTM (1 hora à frente)")
plt.plot(datas_futuro, futuro, "--", label="Projeção próximas 24 h")
plt.title(f"{COIN}: histórico real vs predição (dados novos da API)")
plt.xlabel("Data")
plt.ylabel("Preço (USD)")
plt.legend()
plt.savefig("resultados/06_predicao_dados_novos.png", bbox_inches="tight")
plt.show()

## Conclusões
- Preencher após rodar: comparar o RMSE/MAE da LSTM com a persistência no teste e nos dados novos.
- Indicar qual `window_size` teve o menor erro nos experimentos.
- Observação: em séries de preço de hora em hora, o baseline de persistência costuma ser muito forte, pois o preço muda pouco de uma hora para outra; superar esse baseline é o principal desafio do modelo.